In [ ]:
import os
import math
import torch
import random
import torch.nn.functional as F

from peft import PeftModel

from models import AutoTokenizer, AutoConfig, AutoModelForCausalLM

path = "data/LLaDA-8B-Instruct"

config = AutoConfig.from_pretrained(path, _attn_implementation="eager")

model = AutoModelForCausalLM.from_pretrained(
    path,
    config=config,
    dtype=torch.bfloat16, 
).cuda(0)

model = PeftModel.from_pretrained(model, 'runs/LLaDA-Lora-Full_all_5e-03/checkpoint-final', device_map='auto')
model = model.merge_and_unload()

tokenizer = AutoTokenizer.from_pretrained(path)

pad_id = tokenizer.pad_token_id
eos_id = tokenizer.eos_token_id
mask_id = tokenizer.mask_token_id

In [ ]:
from PIL import Image, ImageDraw, ImageFont
from typing import List, Dict, Optional


def load_font(size: int, bold: bool = False):
    return ImageFont.truetype('data/times_bold.ttf"', size)

def render_token_grid(
    tokens: List[str],
    output_path: str = "token_grid.png",

    # Layout
    max_width: int = 1600,
    scale: int = 2,   # 论文建议 2 或 3；预览可用 1

    # Highlight specific token positions only
    token_bg_colors: Optional[Dict[int, str]] = None,

    # Global colors
    bg_color: str = "#FCFCFA",
    default_token_color: str = "#DCE8F2",
    default_token_outline: str = "#8FA9BF",
    mask_fill: str = "#F3EFE8",
    mask_outline: str = "#B9B1A5",
    text_color: str = "#1F2933",
    mask_text_color: str = "#857C72",

    # Typography / spacing (before scaling)
    font_size: int = 22,
    pad_x: int = 12,
    pad_y: int = 7,
    gap_x: int = 5,
    gap_y: int = 7,
    radius: int = 7,
    margin_left: int = 20,
    margin_top: int = 20,
    margin_right: int = 20,
    margin_bottom: int = 20,
):
    """
    Render a token grid image.

    Args:
        tokens: token list.
        output_path: where to save.
        max_width: canvas width before scaling.
        scale: scale factor for high-resolution export.
        token_bg_colors: dict {token_index: hex_color}, only for specific positions.

    Special token:
        "[M]" will be rendered as a masked token block.
    """

    token_bg_colors = token_bg_colors or {}

    # Apply scaling for high-res export
    max_width = max_width * scale
    font_size = font_size * scale
    pad_x = pad_x * scale
    pad_y = pad_y * scale
    gap_x = gap_x * scale
    gap_y = gap_y * scale
    radius = radius * scale
    margin_left = margin_left * scale
    margin_top = margin_top * scale
    margin_right = margin_right * scale
    margin_bottom = margin_bottom * scale

    font = load_font(font_size, bold=True)

    def text_size(text: str):
        bbox = font.getbbox(text)
        return bbox[2] - bbox[0], bbox[3] - bbox[1]

    # -------- First pass: line wrapping --------
    rows = []
    current_row = []
    x = margin_left
    usable_width = max_width - margin_right

    for idx, tok in enumerate(tokens):
        tw, th = text_size(tok)
        box_w = tw + pad_x * 2

        if x + box_w > usable_width and current_row:
            rows.append(current_row)
            current_row = []
            x = margin_left

        current_row.append((idx, tok, box_w))
        x += box_w + gap_x

    if current_row:
        rows.append(current_row)

    row_h = max(text_size("Mg")[1] + pad_y * 2, 30 * scale)
    total_height = (
        margin_top
        + len(rows) * row_h
        + max(0, len(rows) - 1) * gap_y
        + margin_bottom
    )

    # -------- Create canvas --------
    img = Image.new("RGB", (max_width, total_height), bg_color)
    draw = ImageDraw.Draw(img)

    # -------- Draw rows --------
    y = margin_top
    for row in rows:
        x = margin_left

        for idx, tok, box_w in row:
            is_mask = (tok == "[M]")

            if is_mask:
                fill = mask_fill
                outline = mask_outline
                txt_color = mask_text_color
                border_width = max(2, scale)
            else:
                fill = token_bg_colors.get(idx, default_token_color)
                outline = default_token_outline
                txt_color = text_color
                border_width = max(1, scale)

            box = [x, y, x + box_w, y + row_h]

            # Outer rounded box
            draw.rounded_rectangle(
                box,
                radius=radius,
                fill=fill,
                outline=outline,
                width=border_width,
            )

            # Text
            tw, th = text_size(tok)
            tx = x + (box_w - tw) / 2
            ty = y + (row_h - th) / 2 - scale
            draw.text((tx, ty), tok, font=font, fill=txt_color)

            x += box_w + gap_x

        y += row_h + gap_y

    # Save at high dpi
    img.save(output_path, dpi=(600, 600))
    return img

In [ ]:
def sample(logits, temperature=0.0, top_p=1.0):
    if logits.dim() != 2:
        raise ValueError(f"logits must be 2D [B, V], got shape {logits.shape}")

    if temperature < 0:
        raise ValueError("temperature must be >= 0")

    if not (0 < top_p <= 1.0):
        raise ValueError("top_p must be in (0, 1]")

    if temperature == 0:
        return torch.argmax(logits, dim=-1), F.softmax(logits, dim=-1)

    logits = logits / temperature

    if top_p < 1.0:
        # sort
        sorted_logits, sorted_indices = torch.sort(logits, dim=-1, descending=True)

        # cumulative probs
        sorted_probs = F.softmax(sorted_logits, dim=-1)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

        # mask tokens outside nucleus
        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = False

        # scatter back to original index positions
        indices_to_remove = torch.zeros_like(logits, dtype=torch.bool)
        indices_to_remove.scatter_(dim=-1, index=sorted_indices, src=sorted_indices_to_remove)

        logits = logits.masked_fill(indices_to_remove, float("-inf"))

    probs = F.softmax(logits, dim=-1)
    next_token_id = torch.multinomial(probs, num_samples=1).squeeze(-1)
    return next_token_id, probs

@torch.inference_mode()
def mask_diffusion(input_ids, max_gen_len=256, pred_per_step=4, score_mode='attn'):

    input_len = input_ids.shape[0]
    mask_token = torch.full((max_gen_len, ), mask_id, dtype=torch.long, device=input_ids.device)
    input_ids = torch.cat([input_ids, mask_token], dim=-1)

    answer_mask = input_ids == mask_id

    fig_idx = 0
    for i in range(max_gen_len // pred_per_step):

        print_this_step = False
        bold_idx = []
        bold_tokens = []
        color_dict = {}

        outputs = model(
            input_ids=input_ids.unsqueeze(0), 
            is_causal=False,
            use_cache=False,
            output_attentions=True
        )

        attn_weights = torch.concat(outputs.attentions, dim=0).mean(dim=0).mean(dim=0)
        curr_token_ids, probs = sample(outputs.logits[0], 0.2, 1.0)

        if score_mode == 'entropy':
            scores = torch.sum(probs * torch.log(probs + 1e-8), dim=-1)
            scores[~answer_mask] = -torch.inf
            scores[curr_token_ids == pad_id] = -1000
            pred_positions = torch.topk(scores, k=pred_per_step).indices
        elif score_mode == 'test':
            weights = attn_weights[answer_mask][:, ~answer_mask].sum(dim=1)
            corr = attn_weights[answer_mask, :][:, answer_mask]

            # weights[curr_token_ids[answer_mask] == pad_id] = -1000

            scores = torch.sum(probs * torch.log(probs + 1e-8), dim=-1)[answer_mask]
            # scores[curr_token_ids[answer_mask] == pad_id] = -1000
            max_weights = torch.topk(scores, k=pred_per_step).indices.cpu().numpy().tolist()
            
            selected = []
            for _ in range(pred_per_step):
                coefs = weights.clone()

                if selected:
                    coefs[selected] = -torch.inf
                    coefs = coefs - corr[:, selected].sum(dim=1)

                k = torch.argmax(coefs).item()
                selected.append(k)
            
            pred_positions = torch.where(answer_mask)[0][selected]

            if set(selected) != set(max_weights):
                print_this_step = True

                right = set(selected) - set(max_weights)
                wrong = set(max_weights) - set(selected)

                right = torch.where(answer_mask)[0][list(right)].cpu().numpy().tolist()
                wrong = torch.where(answer_mask)[0][list(wrong)].cpu().numpy().tolist()

                for idx in right:
                    bold_idx.append(idx - input_len)
                    bold_tokens.append(tokenizer.convert_ids_to_tokens(curr_token_ids[idx].item()))
                    color_dict[idx - input_len] = "#A3E635"
                
                for idx in wrong:
                    bold_idx.append(idx - input_len)
                    bold_tokens.append(tokenizer.convert_ids_to_tokens(curr_token_ids[idx].item()))
                    color_dict[idx - input_len] = "#F28C8C"

        answer_mask[pred_positions] = False
        input_ids[pred_positions] = curr_token_ids[pred_positions]

        if print_this_step and '<|endoftext|>' not in bold_tokens:
            fig_idx += 1
            tokens = tokenizer.convert_ids_to_tokens(input_ids[input_len:], skip_special_tokens=False)

            for idx, token in zip(bold_idx, bold_tokens):
                tokens[idx] = token

            replaceed_tokens = []
            for t in tokens:
                t = t.replace('Ġ', '\\n')
                t = t.replace('Ċ', '')
                t = t.replace('<|mask|>', '[M]')
                if t == '<|endoftext|>':
                    continue
                replaceed_tokens.append(t)

            img = render_token_grid(
                replaceed_tokens,
                max_width=1600,
                scale=5,
                token_bg_colors=color_dict,
                output_path=f"fig_{fig_idx}.pdf"
            )

            display(img)

string = "<|im_start|>system\nThe following are multiple choice questions (with answers) about biology. Think step by step and then finish your answer with \"the answer is (X)\" where X is the correct letter choice.\n<|im_end|>\n<|im_start|>user\nQuestion:\nWhich of the following represents an accurate statement concerning arthropods?\nOptions:\nA. They possess an exoskeleton composed primarily of peptidoglycan.\nB. They possess an open circulatory system with a dorsal heart.\nC. They are members of a biologically unsuccessful phylum incapable of exploiting diverse habitats and nutrition sources.\nD. They lack paired, jointed appendages.\nE. N/A\nF. N/A\nG. N/A\nH. N/A\nI. N/A\nJ. N/A\nAnswer: Let's think step by step.<|im_end|>\n<|im_start|>assistant\nPeptidoglycan is known to comprise the plasma membrane of most bacteria, rather than the exoskeleton of arthropods, which is made of chitin, which rules out (A). The answer (C) is false because arthropods are a highly successful phylum. Likewise, arthropods have paired, jointed appendages, which rules out (D). The only remaining option is (B), as arthropods have an open circulatory system with a dorsal tubular heart. The answer is (B).<|im_end|>\n<|im_start|>user\nQuestion:\nWhich of the following would most likely provide examples of mitotic cell divisions?\nOptions:\nA. cross section of muscle tissue\nB. longitudinal section of a shoot tip\nC. longitudinal section of a leaf vein\nD. cross section of a fruit\nE. cross section of a leaf\nF. longitudinal section of a petal\nG. longitudinal section of a seed\nH. cross section of an anther (site of pollen production in a flower)\nAnswer: Let's think step by step.<|im_end|>\n<|im_start|>assistant\n"

input_ids = tokenizer(string, return_tensors='pt')['input_ids'][0].cuda(0)

mask_diffusion(input_ids, max_gen_len=256, pred_per_step=8, score_mode='test')